### This notebook contains all experiments conducted for the project "Trie-Based Execution for Improving Efficiency of PyTerrier Experiments"

In [ ]:
import os
# Set JAVA_HOME for pyTerrier (required for Apache Terrier components)
os.environ['JAVA_HOME'] = r"C:\Users\irene\.jdk\jdk-17.0.16"



In [ ]:
import pyterrier as pt
# from pyterrier_t5 import MonoT5ReRanker
from pyterrier.measures import *
import pyterrier_alpha as pta

In [ ]:
vaswani = pt.datasets.get_dataset("vaswani")
TF_IDF = pt.terrier.Retriever(vaswani.get_index(), wmodel="TF_IDF")
bm25 = pt.terrier.Retriever(vaswani.get_index(), wmodel="BM25")
PL2 = pt.terrier.Retriever(vaswani.get_index(), wmodel="PL2")

Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [ ]:
BM25 = pt.terrier.Retriever(vaswani.get_index(), wmodel="BM25")
rrf = pta.RRFusion(BM25, PL2)
rrf

In [ ]:
pipelines = [BM25%100, rrf, BM25+PL2]

pt.Experiment(
    pipelines,
    vaswani.get_topics(),
    vaswani.get_qrels(),
    ['map','ndcg'],
    plan = 'tree')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,name,map,ndcg
0,(TerrierRetr(BM25) >> RankCutoff(100)),0.272523,0.502299
1,<pyterrier_alpha.fusion.RRFusion object at 0x0...,0.287406,0.612598
2,<pyterrier._ops.Sum object at 0x000001AE2DB02F90>,0.288045,0.611778


In [ ]:
from pprint import pprint

schematic_dict = pt.schematic.transformer_schematic(rrf)
pprint(schematic_dict)          # pretty-print dict
# or: print(schematic_dict)     # raw print

In [ ]:
import inspect
print(inspect.getsource(rrf.schematic))  # should show where "inner_pipelines_mode": "combine" is set
pprint(pt.schematic.transformer_schematic(rrf, default=True))  # show generic/default schematic path

In [ ]:
pipeline = BM25 % 100 >> TF_IDF
pipeline
pipeline[2]

In [ ]:
pipelines = [BM25%100, BM25%100>>PL2, BM25%100>>PL2%90>>TF_IDF%50>>TF_IDF%10>>PL2, BM25%100>>PL2%20>>TF_IDF]

pt.Experiment(
    [
            bm25 % 100,
            bm25 % 100 >> PL2,
            bm25 % 100 >> PL2 % 10 >> TF_IDF>>PL2,
            bm25 % 100 >> PL2 % 20 >> TF_IDF,
        ],
    vaswani.get_topics(),
    vaswani.get_qrels(),
    ['map','ndcg'],
    plan = 'tree', verbose = True)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,name,map,ndcg
0,(TerrierRetr(BM25) >> RankCutoff(100)),0.272523,0.502299
1,(TerrierRetr(BM25) >> RankCutoff(100) >> Terri...,0.255893,0.487972
2,(TerrierRetr(BM25) >> RankCutoff(100) >> Terri...,0.156815,0.279109
3,(TerrierRetr(BM25) >> RankCutoff(100) >> Terri...,0.192460,0.340603


In [ ]:
pt.Experiment(
    pipelines,
    vaswani.get_topics(),
    vaswani.get_qrels(),
    ['map','ndcg'],
    plan = 'tree')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,name,map,ndcg
0,(TerrierRetr(BM25) >> RankCutoff(100)),0.272523,0.502299
1,(TerrierRetr(BM25) >> RankCutoff(100) >> Terri...,0.255893,0.487972
2,(TerrierRetr(BM25) >> RankCutoff(100) >> Terri...,0.160542,0.285238
3,(TerrierRetr(BM25) >> RankCutoff(100) >> Terri...,0.192460,0.340603


In [ ]:
from pprint import pprint
from pyterrier._evaluation._exec_tree import decompose_pipelines, TransformerRadixTree
retr_systems = [BM25%10, BM25%10>>PL2]
# Step 1: Decompose
decomposed = decompose_pipelines(retr_systems)

# Step 2: Create tree
tree = TransformerRadixTree()

# Step 3: Insert each pipeline
for sysid, system in enumerate(decomposed):
    key = tuple(system)
    tree.insert(key, sysid)

# Step 4: Now generate and print schematic
from pprint import pprint
schematic = pt.schematic.radix_tree_schematic(tree, input_columns=["qid", "query"])
pprint(schematic)



In [ ]:
retr_systems = [BM25%10, BM25>> TF_IDF, PL2, TF_IDF]

pt.Experiment(
    retr_systems,
    vaswani.get_topics(),
    vaswani.get_qrels(),
    ['map','ndcg'],
    plan = 'tree')

In [ ]:
from IPython.display import HTML



# Draw just BM25 by itself (not part of a tree)
HTML(pt.schematic.draw(BM25))

# For Google Colab Experiments

In [ ]:
import pyterrier as pt
from pyterrier._evaluation._trie import decompose_pipelines, RadixTree
from pyterrier.measures import *
from pyterrier_t5 import MonoT5ReRanker, DuoT5ReRanker

In [ ]:
index = pt.IndexFactory.of(pt.get_dataset('msmarco_passage').get_index('terrier_stemmed_text'), memory=False)
bm25 = pt.terrier.Retriever(index, metadata=['docno', 'text'], wmodel='BM25', verbose=True)
monoT5 = MonoT5ReRanker()
duoT5 = DuoT5ReRanker()
dataset = pt.get_dataset("irds:msmarco-passage/trec-dl-2019/judged")

# please replace the above transformers with the ones below for quicker computation, if you wish to
# vaswani = pt.datasets.get_dataset("vaswani")
# TF_IDF = pt.terrier.Retriever(vaswani.get_index(), wmodel="TF_IDF")
# BM25 = pt.terrier.Retriever(vaswani.get_index(), wmodel="BM25")
# PL2 = pt.terrier.Retriever(vaswani.get_index(), wmodel="PL2")

## RQ1:  Does the tree execution have comparable execution time to the linear execution when all pipelines share a common prefix

In [ ]:
%%time
pipelines_1 = [bm25 % k1 >> monoT5  for k1 in [10,20]]
pt.Experiment(
        pipelines_1,
        dataset.get_topics(),
        dataset.get_qrels(),
        [nDCG@10],
        precompute_prefix=False,
        plan = 'linear'
    )

In [ ]:
%%time
pipelines_1 = [bm25 % k1 >> monoT5  for k1 in [10,20]]
pt.Experiment(
        pipelines_1,
        dataset.get_topics(),
        dataset.get_qrels(),
        [nDCG@10],
        precompute_prefix=True,
        plan = 'linear'
    )

In [ ]:
%%time
pipelines_1 = [bm25 % k1 >> monoT5  for k1 in [10,20]]
pt.Experiment(
        pipelines_1,
        dataset.get_topics(),
        dataset.get_qrels(),
        [nDCG@10],
        plan = 'tree'
    )

## RQ2: How does prefix depth impact improvements in execution time?

In [ ]:
%%time
pipelines_2 = [bm25 % k1 >> monoT5 % k2 >> duoT5  for k1 in [10,20] for k2 in [5,10]]
pt.Experiment(
        pipelines_2,
        dataset.get_topics(),
        dataset.get_qrels(),
        [nDCG@10],
        precompute_prefix=False,
        plan = 'linear'
    )

In [ ]:
%%time
pipelines_2 = [bm25 % k1 >> monoT5 % k2 >> duoT5  for k1 in [10,20] for k2 in [5,10]]
pt.Experiment(
        pipelines_2,
        dataset.get_topics(),
        dataset.get_qrels(),
        [nDCG@10],
        precompute_prefix=True,
        plan = 'linear'
    )

In [ ]:
%%time
pipelines_2 = [bm25 % k1 >> monoT5 % k2 >> duoT5  for k1 in [10,20] for k2 in [5,10]]
pt.Experiment(
        pipelines_2,
        dataset.get_topics(),
        dataset.get_qrels(),
        [nDCG@10],
        plan = 'tree'
    )

## RQ3:  Does the execution time improve using tree execution when there are two distinct common prefixes?


In [ ]:
%%time
pipelines_3 = [t % k1 >> monoT5 % k2 >> duoT5 for t in [bm25,tf_idf] for k1 in [10,20] for k2 in [5,10]]
pt.Experiment(
        pipelines_3,
        dataset.get_topics(),
        dataset.get_qrels(),
        [nDCG@10],
        precompute_prefix=False,
        plan = 'linear'
    )

In [ ]:
%%time
pipelines_3 = [t % k1 >> monoT5 % k2 >> duoT5 for t in [bm25,tf_idf] for k1 in [10,20] for k2 in [5,10]]
pt.Experiment(
        pipelines_3,
        dataset.get_topics(),
        dataset.get_qrels(),
        [nDCG@10],
        precompute_prefix=True,
        plan = 'linear'
    )

In [ ]:
%%time
pipelines_3 = [t % k1 >> monoT5 % k2 >> duoT5 for t in [bm25,tf_idf] for k1 in [10,20] for k2 in [5,10]]
pt.Experiment(
        pipelines_3,
        dataset.get_topics(),
        dataset.get_qrels(),
        [nDCG@10],
        plan = 'tree'
    )

## For real world experiment:

In [ ]:
# Load the vaswani dataset using PyTerrier's native dataset functionality
vaswani = pt.datasets.get_dataset("vaswani")
datasets = pt.get_dataset('irds:vaswani')
tf_idf = pt.terrier.Retriever(vaswani.get_index(), wmodel="TF_IDF")
bm25 = pt.terrier.Retriever(vaswani.get_index(), wmodel="BM25")
monoT5 = MonoT5ReRanker()
duoT5 = DuoT5ReRanker()

In [ ]:
import pandas as pd
labels_df = vaswani.get_qrels()
labels_df['label'] = labels_df['label'].astype(float)

def filter_func(run):
  # Merge on both 'qid' and 'docno' to correctly associate labels
  # and ensure 'qid' column is preserved without renaming issues.
  run_with_labels = pd.merge(run, labels_df[['qid', 'docno', 'label']], on = ["qid", "docno"], how='inner')

  # Filter documents based on the label
  filtered_run = run_with_labels[run_with_labels["label"] >=  0.5]

  # Drop the 'rank' column. Use errors='ignore' to prevent error if 'rank' is already gone.
  filtered_run = filtered_run.drop(columns = ["rank"], errors='ignore')

  # Add new ranks based on the scores of the filtered documents
  return pt.model.add_ranks(filtered_run)

filter_docs = pt.apply.generic(filter_func)
# RQ: Is it better to filter after one stage of reranking or two?
rq2_v1 = bm25 >> pt.text.get_text(datasets,"text")>>  monoT5  >> filter_docs % 10 >> duoT5
rq2_v2 = bm25 >> pt.text.get_text(datasets,"text") >> monoT5  % 10 >> duoT5
rq2_v3 = tf_idf >> pt.text.get_text(datasets,"text") >> monoT5  >> filter_docs %10>> duoT5
rq2_v4 = tf_idf >> pt.text.get_text(datasets,"text") >> monoT5  % 10 >> duoT5

In [ ]:
%%time
pt.Experiment(
    [rq2_v1, rq2_v2, rq2_v3, rq2_v4],
    vaswani.get_topics(),
    vaswani.get_qrels(),
    [nDCG@10],
    precompute_prefix=False,
    plan = 'linear'

)

In [ ]:
%%time
pt.Experiment(
    [rq2_v1, rq2_v2, rq2_v3, rq2_v4],
    vaswani.get_topics(),
    vaswani.get_qrels(),
    [nDCG@10],
    precompute_prefix=True,
    plan = 'linear'

)

In [ ]:
%%time
pt.Experiment(
    [rq2_v1, rq2_v2, rq2_v3, rq2_v4],
    vaswani.get_topics(),
    vaswani.get_qrels(),
    [nDCG@10],
    plan = 'tree'

)